# 🔒 Pipeline Completo de Análisis de Seguridad

**Curso:** Ciberseguridad (ICC610) - 2026  
**Objetivo:** Ejecutar el pipeline completo de análisis de seguridad y realizar el análisis cuantitativo y cualitativo, todo desde un solo notebook.

## ¿Qué hace este notebook?

| Fase | Descripción | Herramienta |
|------|-------------|-------------|
| **1** | Configuración e imports | Python |
| **2** | Clonar repositorios | Git |
| **3** | Generar SBOM (lista de dependencias) | Syft |
| **4** | Escanear vulnerabilidades en dependencias | Grype |
| **5** | Análisis estático del código fuente | CodeQL |
| **6** | Generar reporte consolidado | Python |
| **7** | Análisis cuantitativo de dependencias | Pandas |
| **8** | Análisis cuantitativo de vulnerabilidades | Pandas |
| **9** | Análisis cuantitativo de CodeQL | Pandas |
| **10** | Análisis de configuraciones CI/CD | PyYAML / regex |
| **11** | Resumen ejecutivo | Python |
| **12** | Exportar a CSV | Pandas |
| **13** | Relación con casos de referencia | Pandas |
| **14** | Interpretación y riesgos del ecosistema | Pandas |

> ⚠️ **Nota:** Ejecuta las celdas en orden. La primera ejecución puede tardar 10-20 minutos (clonado + CodeQL).

---

## 1. ⚙️ Configuración e Imports

In [67]:
import json
import subprocess
import sys
import os
import shutil
import pandas as pd
from pathlib import Path
from collections import Counter
from datetime import datetime

# Detectar PROJECT_ROOT automaticamente (devcontainer, Codespaces, local)
_search = Path(os.getcwd())
while not (_search / "data" / "config.json").exists():
    if _search.parent == _search:
        _search = Path(os.getcwd())
        break
    _search = _search.parent
PROJECT_ROOT = _search

REPOS_DIR = PROJECT_ROOT / "data" / "repos"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
CONFIG_FILE = PROJECT_ROOT / "data" / "config.json"

REPOS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

print("=" * 60)
print("  PIPELINE COMPLETO DE ANALISIS DE SEGURIDAD")
print("=" * 60)
print(f"  Proyecto:   {PROJECT_ROOT}")
print(f"  Repos:      {REPOS_DIR}")
print(f"  Resultados: {RESULTS_DIR}")
print(f"  Config:     {CONFIG_FILE}")
print(f"  Fecha:      {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

print("\nVerificando herramientas instaladas...")
tools = {"syft": "syft version", "grype": "grype version", "codeql": "codeql version"}
tools_ok = True
for tool, cmd in tools.items():
    result = subprocess.run(cmd.split(), capture_output=True, text=True)
    version = result.stdout.strip().split("\n")[0] if result.returncode == 0 else "NO ENCONTRADO"
    status = "OK" if result.returncode == 0 else "ERROR"
    print(f"  [{status}] {tool:10} -> {version}")
    if result.returncode != 0:
        tools_ok = False

if tools_ok:
    print("\nTodas las herramientas estan disponibles.")
else:
    print("\nAlgunas herramientas no estan disponibles. Algunos pasos podrian fallar.")


  PIPELINE COMPLETO DE ANALISIS DE SEGURIDAD
  Proyecto:   /workspaces/sbom-vuln-analysis
  Repos:      /workspaces/sbom-vuln-analysis/data/repos
  Resultados: /workspaces/sbom-vuln-analysis/data/results
  Config:     /workspaces/sbom-vuln-analysis/data/config.json
  Fecha:      2026-04-26 02:05:12

Verificando herramientas instaladas...
  [OK] syft       -> Application:   syft
  [OK] grype      -> Application:         grype
  [OK] codeql     -> CodeQL command-line toolchain release 2.25.1.

Todas las herramientas estan disponibles.


### 1.1 Configurar repositorios a analizar

Modifica la lista `REPOSITORIES` para cambiar qué repositorios analizar.  
El `config.json` se actualizará automáticamente.

In [68]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIGURACIÓN: Repositorios a analizar                     ║
# ║  Este bloque lee la configuración actual. Para cambiarla,   ║
# ║  edita directamente el archivo data/config.json             ║
# ╚══════════════════════════════════════════════════════════════╝

if CONFIG_FILE.exists():
    with open(CONFIG_FILE, "r") as f:
        config = json.load(f)
    print(f"✅ Cargando configuración existente desde {CONFIG_FILE}")
else:
    print(f"⚠️ No se encontró {CONFIG_FILE}. Creando configuración por defecto...")
    config = {
        "repos_dir": "data/repos",
        "output_dir": "data/results",
        "repositories": ["https://github.com/pallets/flask"],
        "organizations": [],
        "clone_options": {
            "max_inactive_days": 30,
            "skip_archived": True,
            "skip_forks": True,
            "max_repos": 50
        },
        "tools": {
            "syft": {"enabled": True, "output_format": "json"},
            "grype": {"enabled": True, "output_format": "json", "update_db": True},
            "codeql": {
                "enabled": True,
                "output_format": "json",
                "supported_languages": ["python", "javascript", "java", "cpp", "csharp"]
            }
        },
        "concurrency": {
            "max_workers": 4,
            "enabled": True
        }
    }
    with open(CONFIG_FILE, "w") as f:
        json.dump(config, f, indent=4)

REPOSITORIES = config.get("repositories", [])
ORGANIZATIONS = config.get("organizations", [])

print("\n�� Configuración actual:")
print(f"  Repositorios individuales: {len(REPOSITORIES)}")
for r in REPOSITORIES:
    print(f"    → {r}")
if ORGANIZATIONS:
    print(f"  Organizaciones: {len(ORGANIZATIONS)}")
    for o in ORGANIZATIONS:
        print(f"    → {o}")


✅ Cargando configuración existente desde /workspaces/sbom-vuln-analysis/data/config.json

�� Configuración actual:
  Repositorios individuales: 2
    → https://github.com/apache/superset
    → https://github.com/apache/echarts


---

## 2. 📥 Clonar Repositorios

Descarga los repositorios configurados en `data/repos/`.

In [69]:
print("═" * 60)
print("  [1/5] 📥 CLONANDO REPOSITORIOS")
print("═" * 60)

result = subprocess.run(
    ["uv", "run", "python", "main.py", "clone"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr)
else:
    # Listar repos clonados
    repos = [d.name for d in REPOS_DIR.iterdir() if d.is_dir()]
    print(f"\n📂 Repositorios disponibles en data/repos/: {repos}")

════════════════════════════════════════════════════════════
  [1/5] 📥 CLONANDO REPOSITORIOS
════════════════════════════════════════════════════════════


═══════════════════════════════════════
  CLONADOR DE REPOSITORIOS
═══════════════════════════════════════

📋 2 repositorio(s) individual(es) configurado(s)

🔄 Clonando 2 repositorio(s) — modo paralelo (5 workers)...
  ↓ Clonando: superset
  ↓ Clonando: echarts
  ✓ Clonado: echarts
  ✓ Clonado: superset
                                                 
              Resumen de Clonación               
┏━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Repositorio ┃ Estado    ┃ Ruta                ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ echarts     │ ✓ Clonado │ data/repos/echarts  │
│ superset    │ ✓ Clonado │ data/repos/superset │
└─────────────┴───────────┴─────────────────────┘

✓ 2 exitosos  ✗ 0 errores
📄 Log guardado en: data/results/clone-log.json


📂 Repositorios disponibles en data/repos/: ['echarts', 'superset']


---

## 3. 📦 Generar SBOMs (Syft)

Genera la lista de componentes y dependencias de cada repositorio usando **Syft**.

In [70]:
print("═" * 60)
print("  [2/5] 📦 GENERANDO SBOMs (SYFT)")
print("═" * 60)

result = subprocess.run(
    ["uv", "run", "python", "main.py", "sbom"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr)
else:
    sbom_files = sorted(RESULTS_DIR.glob("*-sbom.json"))
    print(f"\n✅ {len(sbom_files)} SBOM(s) generados")
    for f in sbom_files:
        size = f.stat().st_size / 1024
        print(f"   📄 {f.name} ({size:.1f} KB)")

════════════════════════════════════════════════════════════
  [2/5] 📦 GENERANDO SBOMs (SYFT)
════════════════════════════════════════════════════════════
═══════════════════════════════
GENERADOR DE SBOMs - Syft
═══════════════════════════════

📦 Procesando 2 repositorio(s) — modo paralelo (5 workers)...

Generando SBOM para: echarts

Generando SBOM para: superset
✓ SBOM generado: data/results/echarts-sbom.json
✓ SBOM generado: data/results/superset-sbom.json

Resumen guardado en: data/results/sbom-summary.json


✅ 2 SBOM(s) generados
   📄 echarts-sbom.json (35.6 KB)
   📄 superset-sbom.json (8008.2 KB)


---

## 4. 🔓 Escanear Vulnerabilidades (Grype)

Busca CVEs conocidos en las dependencias detectadas usando **Grype**.

> ℹ️ La primera ejecución descarga la base de datos de vulnerabilidades (~2 min).

In [71]:
print("═" * 60)
print("  [3/5] 🔓 ESCANEANDO VULNERABILIDADES (GRYPE)")
print("═" * 60)

result = subprocess.run(
    ["uv", "run", "python", "main.py", "grype"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT),
    timeout=600
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr)
else:
    grype_files = sorted(RESULTS_DIR.glob("*-grype.json"))
    print(f"\n✅ {len(grype_files)} escaneo(s) completados")
    for f in grype_files:
        with open(f) as fh:
            data = json.load(fh)
        n_vulns = len(data.get("matches", []))
        print(f"   🔓 {f.stem}: {n_vulns} vulnerabilidades")

════════════════════════════════════════════════════════════
  [3/5] 🔓 ESCANEANDO VULNERABILIDADES (GRYPE)
════════════════════════════════════════════════════════════
═══════════════════════════════
ESCANER DE VULNERABILIDADES - Grype
═══════════════════════════════

Actualizando base de datos de Grype...

🔓 Escaneando 2 repositorio(s) — modo paralelo (5 workers)...

Escaneando vulnerabilidades: echarts

Escaneando vulnerabilidades: superset
✓ Escaneo completado: 0 vulnerabilidades encontradas
✓ Escaneo completado: 26 vulnerabilidades encontradas
             Resumen de Escaneo             
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Repositorio ┃ Vulnerabilidades ┃ Estado  ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ echarts     │ 0                │ success │
│ superset    │ 26               │ success │
└─────────────┴──────────────────┴─────────┘

Resumen guardado en: data/results/grype-summary.json


✅ 2 escaneo(s) completados
   🔓 echarts-grype: 0 vulnerabilidades
   🔓 supe

---

## 5. 🔍 Análisis Estático de Código (CodeQL)

Encuentra vulnerabilidades en el código fuente (SQL injection, XSS, etc.) usando **CodeQL**.

> ⏳ **Este paso puede tardar 5-20 minutos** por repositorio. Si deseas saltarlo, comenta la celda y continúa.

In [ ]:
print("═" * 60)
print("  [4/5] 🔍 ANÁLISIS ESTÁTICO (CODEQL)")
print("═" * 60)
print("⏳ Este paso puede tardar varios minutos...\n")

result = subprocess.run(
    ["uv", "run", "python", "main.py", "codeql"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT),
    timeout=3600
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
else:
    codeql_files = sorted(RESULTS_DIR.glob("*-codeql.json"))
    print(f"\n✅ {len(codeql_files)} análisis completados")
    for f in codeql_files:
        with open(f) as fh:
            data = json.load(fh)
        n_findings = data.get("total", len(data.get("findings", [])))
        print(f"   🔍 {f.stem}: {n_findings} hallazgos")

════════════════════════════════════════════════════════════
  [4/5] 🔍 ANÁLISIS ESTÁTICO (CODEQL)
════════════════════════════════════════════════════════════
⏳ Este paso puede tardar varios minutos...



---

## 6. 📊 Generar Reporte Consolidado

In [ ]:
print("═" * 60)
print("  [5/5] 📊 GENERANDO REPORTE CONSOLIDADO")
print("═" * 60)

result = subprocess.run(
    ["uv", "run", "python", "main.py", "report"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr)
else:
    report_file = RESULTS_DIR / "consolidated-report.json"
    if report_file.exists():
        with open(report_file) as f:
            report = json.load(f)
        print("\n📋 Contenido del reporte:")
        print(json.dumps(report, indent=2))

print("\n" + "═" * 60)
print("  ✅ PIPELINE COMPLETO EJECUTADO")
print("═" * 60)

# Resumen de archivos generados
print("\n📂 Archivos generados en data/results/:")
for f in sorted(RESULTS_DIR.iterdir()):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f"   {'📄' if size < 100 else '📦'} {f.name:40} ({size:.1f} KB)")

════════════════════════════════════════════════════════════
  [5/5] 📊 GENERANDO REPORTE CONSOLIDADO
════════════════════════════════════════════════════════════


═══════════════════════════════
GENERADOR DE REPORTES
═══════════════════════════════
           SBOMs Generados            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Repositorio              ┃ Estado  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ tooling-trusted-releases │ success │
│ airflow                  │ success │
└──────────────────────────┴─────────┘
                Vulnerabilidades (Grype)                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Repositorio              ┃ Vulnerabilidades ┃ Estado  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ tooling-trusted-releases │ 1                │ success │
│ airflow                  │ 19               │ success │
└──────────────────────────┴──────────────────┴─────────┘
                Análisis Estático (CodeQL)                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Repositorio              ┃ Lenguajes          ┃ Estado  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━

---

# 📊 ANÁLISIS CUANTITATIVO

A partir de aquí se analizan cuantitativamente los resultados generados por el pipeline.

---

## 7. 📦 Análisis de Dependencias (SBOM)

Analizamos los componentes de software detectados por Syft en cada repositorio.

In [ ]:
# Cargar todos los SBOMs
sbom_files = sorted(RESULTS_DIR.glob("*-sbom.json"))
grype_files = sorted(RESULTS_DIR.glob("*-grype.json"))
codeql_files = sorted(RESULTS_DIR.glob("*-codeql.json"))

print("═" * 60)
print("  📂 ARCHIVOS DE RESULTADOS ENCONTRADOS")
print("═" * 60)
print(f"  📦 SBOMs:           {len(sbom_files)} archivo(s)")
print(f"  🔓 Grype (vulns):   {len(grype_files)} archivo(s)")
print(f"  🔍 CodeQL:          {len(codeql_files)} archivo(s)")
print("═" * 60)

if not sbom_files and not grype_files:
    print("\n⚠️  No se encontraron resultados. Ejecuta las celdas anteriores del pipeline.")

════════════════════════════════════════════════════════════
  📂 ARCHIVOS DE RESULTADOS ENCONTRADOS
════════════════════════════════════════════════════════════
  📦 SBOMs:           2 archivo(s)
  🔓 Grype (vulns):   2 archivo(s)
  🔍 CodeQL:          1 archivo(s)
════════════════════════════════════════════════════════════


In [ ]:
# Cargar todos los SBOMs y extraer dependencias
all_dependencies = []

for sbom_file in sbom_files:
    repo_name = sbom_file.stem.replace("-sbom", "")
    
    with open(sbom_file) as f:
        data = json.load(f)
    
    artifacts = data.get("artifacts", [])
    
    for art in artifacts:
        all_dependencies.append({
            "repo": repo_name,
            "name": art.get("name", "N/A"),
            "version": art.get("version", "N/A"),
            "type": art.get("type", "N/A"),
            "language": art.get("language", "N/A"),
            "licenses": ", ".join(
                [lic.get("value", "N/A") for lic in art.get("licenses", [])]
            ) or "No especificada",
        })

df_deps = pd.DataFrame(all_dependencies)

if not df_deps.empty:
    print(f"\n📦 Total de dependencias detectadas: {len(df_deps)}")
    print(f"📁 Repositorios analizados: {df_deps['repo'].nunique()}")
    print(f"\n--- Primeras 10 dependencias ---")
    display(df_deps.head(10))
else:
    print("⚠️  No se encontraron dependencias en los SBOMs.")


📦 Total de dependencias detectadas: 5746
📁 Repositorios analizados: 2

--- Primeras 10 dependencias ---


,repo,name,version,type,language,licenses
0,airflow,./.github/actions/breeze,UNKNOWN,github-action,,No especificada
1,airflow,./.github/actions/breeze,UNKNOWN,github-action,,No especificada
2,airflow,./.github/actions/breeze,UNKNOWN,github-action,,No especificada
3,airflow,./.github/actions/breeze,UNKNOWN,github-action,,No especificada
4,airflow,./.github/actions/breeze,UNKNOWN,github-action,,No especificada
5,airflow,./.github/actions/breeze,UNKNOWN,github-action,,No especificada
6,airflow,./.github/actions/breeze,UNKNOWN,github-action,,No especificada
7,airflow,./.github/actions/breeze,UNKNOWN,github-action,,No especificada
8,airflow,./.github/actions/breeze,UNKNOWN,github-action,,No especificada
9,airflow,./.github/actions/breeze,UNKNOWN,github-action,,No especificada


### 7.1 Dependencias por tipo de paquete

In [ ]:
if not df_deps.empty:
    print("\n📊 Distribución de dependencias por tipo de paquete:\n")
    type_counts = df_deps["type"].value_counts()
    
    max_count = type_counts.max()
    for pkg_type, count in type_counts.items():
        bar_len = int(count / max_count * 40)
        bar = "█" * bar_len
        pct = count / len(df_deps) * 100
        print(f"  {pkg_type:20} │ {bar} {count} ({pct:.1f}%)")
    
    print(f"\n  {'TOTAL':20} │ {len(df_deps)}")


📊 Distribución de dependencias por tipo de paquete:

  npm                  │ ████████████████████████████████████████ 4319 (75.2%)
  python               │ ██████████ 1167 (20.3%)
  github-action        │ █ 171 (3.0%)
  go-module            │  48 (0.8%)
  github-action-workflow │  33 (0.6%)
  java-archive         │  8 (0.1%)

  TOTAL                │ 5746


### 7.2 Dependencias por repositorio

In [ ]:
if not df_deps.empty:
    print("\n📊 Cantidad de dependencias por repositorio:\n")
    repo_counts = df_deps.groupby("repo").size().sort_values(ascending=False)
    
    max_count = repo_counts.max()
    for repo, count in repo_counts.items():
        bar_len = int(count / max_count * 40)
        bar = "█" * bar_len
        print(f"  {repo:30} │ {bar} {count}")
    
    print(f"\n  Promedio por repositorio: {len(df_deps) / df_deps['repo'].nunique():.1f}")


📊 Cantidad de dependencias por repositorio:

  airflow                        │ ████████████████████████████████████████ 5443
  tooling-trusted-releases       │ ██ 303

  Promedio por repositorio: 2873.0


### 7.3 Licencias más comunes

In [ ]:
if not df_deps.empty:
    print("\n📜 Top 10 licencias más comunes:\n")
    license_counts = df_deps["licenses"].value_counts().head(10)
    
    for lic, count in license_counts.items():
        pct = count / len(df_deps) * 100
        bar = "█" * max(1, int(pct))
        print(f"  {lic:35} │ {bar} {count} ({pct:.1f}%)")


📜 Top 10 licencias más comunes:

  No especificada                     │ █████████████████████████████████████████████████████████████████████████████████████████████████ 5617 (97.8%)
  MIT                                 │ █ 82 (1.4%)
  ISC                                 │ █ 33 (0.6%)
  Apache-2.0                          │ █ 6 (0.1%)
  BSD-3-Clause                        │ █ 6 (0.1%)
  (MPL-2.0 OR Apache-2.0)             │ █ 1 (0.0%)
  Unlicense                           │ █ 1 (0.0%)


### 7.4 Tabla resumen de dependencias

In [ ]:
if not df_deps.empty:
    summary_deps = df_deps.groupby("repo").agg(
        total_deps=("name", "count"),
        tipos_unicos=("type", "nunique"),
        paquetes_unicos=("name", "nunique"),
    ).reset_index()
    
    print("\n📋 Resumen de dependencias por repositorio:\n")
    display(summary_deps)


📋 Resumen de dependencias por repositorio:



,repo,total_deps,tipos_unicos,paquetes_unicos
0,airflow,5443,6,2389
1,tooling-trusted-releases,303,3,289


---

## 8. 🔓 Análisis de Vulnerabilidades (Grype)

Analizamos las vulnerabilidades detectadas en las dependencias de cada repositorio.

In [ ]:
# Cargar todos los resultados de Grype
all_vulns = []

for grype_file in grype_files:
    repo_name = grype_file.stem.replace("-grype", "")
    
    with open(grype_file) as f:
        data = json.load(f)
    
    matches = data.get("matches", [])
    
    for match in matches:
        vuln = match.get("vulnerability", {})
        artifact = match.get("artifact", {})
        
        all_vulns.append({
            "repo": repo_name,
            "vuln_id": vuln.get("id", "N/A"),
            "severity": vuln.get("severity", "Unknown"),
            "description": vuln.get("description", "N/A")[:100],
            "package": artifact.get("name", "N/A"),
            "version": artifact.get("version", "N/A"),
            "pkg_type": artifact.get("type", "N/A"),
            "fix_state": vuln.get("fix", {}).get("state", "N/A"),
            "fix_versions": ", ".join(vuln.get("fix", {}).get("versions", [])),
            "data_source": vuln.get("dataSource", "N/A"),
        })

df_vulns = pd.DataFrame(all_vulns)

if not df_vulns.empty:
    print(f"\n🔓 Total de vulnerabilidades detectadas: {len(df_vulns)}")
    print(f"📁 Repositorios con vulnerabilidades: {df_vulns['repo'].nunique()}")
    print(f"📦 Paquetes afectados: {df_vulns['package'].nunique()}")
    print(f"\n--- Todas las vulnerabilidades ---")
    display(df_vulns[["repo", "vuln_id", "severity", "package", "version", "fix_state", "fix_versions"]])
else:
    print("⚠️  No se encontraron vulnerabilidades (o no se ejecutó Grype).")


🔓 Total de vulnerabilidades detectadas: 20
📁 Repositorios con vulnerabilidades: 2
📦 Paquetes afectados: 9

--- Todas las vulnerabilidades ---


,repo,vuln_id,severity,package,version,fix_state,fix_versions
0,airflow,GHSA-53mr-6c8q-9789,High,litellm,1.82.6,fixed,1.83.0
1,airflow,GHSA-jjhc-v7c2-5hh6,Critical,litellm,1.82.6,fixed,1.83.0
2,airflow,GHSA-mh2q-q3fh-2475,High,go.opentelemetry.io/otel,v1.39.0,fixed,1.41.0
3,airflow,GHSA-3v7f-55p6-f55p,Medium,picomatch,4.0.3,fixed,4.0.4
4,airflow,GHSA-qx2v-qp2m-jg93,Medium,postcss,8.5.6,fixed,8.5.10
5,airflow,GHSA-qx2v-qp2m-jg93,Medium,postcss,8.5.8,fixed,8.5.10
6,airflow,GHSA-qx2v-qp2m-jg93,Medium,postcss,8.5.8,fixed,8.5.10
7,airflow,GHSA-qx2v-qp2m-jg93,Medium,postcss,8.5.8,fixed,8.5.10
8,airflow,GHSA-qx2v-qp2m-jg93,Medium,postcss,8.5.9,fixed,8.5.10
9,airflow,GHSA-qx2v-qp2m-jg93,Medium,postcss,8.5.9,fixed,8.5.10


### 8.1 Distribución de vulnerabilidades por severidad

In [ ]:
if not df_vulns.empty:
    print("\n🎯 Distribución por severidad:\n")
    
    severity_order = ["Critical", "High", "Medium", "Low", "Negligible", "Unknown"]
    severity_colors = {
        "Critical": "🔴",
        "High": "🟠",
        "Medium": "🟡",
        "Low": "🟢",
        "Negligible": "⚪",
        "Unknown": "❓"
    }
    
    sev_counts = df_vulns["severity"].value_counts()
    
    for sev in severity_order:
        if sev in sev_counts.index:
            count = sev_counts[sev]
            pct = count / len(df_vulns) * 100
            icon = severity_colors.get(sev, "")
            bar = "█" * max(1, int(pct / 2))
            print(f"  {icon} {sev:12} │ {bar} {count} ({pct:.1f}%)")
    
    print(f"\n  Total: {len(df_vulns)} vulnerabilidades")


🎯 Distribución por severidad:

  🔴 Critical     │ █████ 2 (10.0%)
  🟠 High         │ █████████████████ 7 (35.0%)
  🟡 Medium       │ ███████████████████████████ 11 (55.0%)

  Total: 20 vulnerabilidades


### 8.2 Vulnerabilidades por repositorio y severidad

In [ ]:
if not df_vulns.empty:
    print("\n📊 Vulnerabilidades por repositorio y severidad:\n")
    
    pivot = df_vulns.pivot_table(
        index="repo",
        columns="severity",
        values="vuln_id",
        aggfunc="count",
        fill_value=0
    )
    
    # Reordenar columnas
    cols = [c for c in severity_order if c in pivot.columns]
    pivot = pivot[cols]
    pivot["TOTAL"] = pivot.sum(axis=1)
    pivot = pivot.sort_values("TOTAL", ascending=False)
    
    display(pivot)


📊 Vulnerabilidades por repositorio y severidad:



severity,Critical,High,Medium,TOTAL
repo,,,,
airflow,2,7,10,19
tooling-trusted-releases,0,0,1,1


### 8.3 Paquetes más vulnerables

In [ ]:
if not df_vulns.empty:
    print("\n📦 Paquetes con más vulnerabilidades:\n")
    
    pkg_vulns = df_vulns.groupby(["package", "version"]).agg(
        total_vulns=("vuln_id", "count"),
        critical=("severity", lambda x: (x == "Critical").sum()),
        high=("severity", lambda x: (x == "High").sum()),
        medium=("severity", lambda x: (x == "Medium").sum()),
        low=("severity", lambda x: (x == "Low").sum()),
    ).reset_index().sort_values("total_vulns", ascending=False)
    
    display(pkg_vulns)
    
    # Gráfico ASCII
    print("\n📊 Gráfico de vulnerabilidades por paquete:\n")
    max_v = pkg_vulns["total_vulns"].max()
    for _, row in pkg_vulns.iterrows():
        name = f"{row['package']}@{row['version']}"
        bar_len = int(row['total_vulns'] / max_v * 35)
        bar = "🔴" * int(row['critical']) + "🟠" * int(row['high']) + "🟡" * int(row['medium']) + "🟢" * int(row['low'])
        print(f"  {name:30} │ {bar} {int(row['total_vulns'])}")


📦 Paquetes con más vulnerabilidades:



,package,version,total_vulns,critical,high,medium,low
3,litellm,1.82.6,5,2,3,0,0
8,postcss,8.5.8,3,0,0,3,0
6,pip,26.0.1,2,0,0,2,0
9,postcss,8.5.9,2,0,0,2,0
5,picomatch,4.0.3,2,0,1,1,0
0,follow-redirects,1.15.11,1,0,0,1,0
1,go.opentelemetry.io/otel,v1.39.0,1,0,1,0,0
2,liquidjs,10.25.5,1,0,1,0,0
4,lxml,6.0.2,1,0,1,0,0
7,postcss,8.5.6,1,0,0,1,0



📊 Gráfico de vulnerabilidades por paquete:

  litellm@1.82.6                 │ 🔴🔴🟠🟠🟠 5
  postcss@8.5.8                  │ 🟡🟡🟡 3
  pip@26.0.1                     │ 🟡🟡 2
  postcss@8.5.9                  │ 🟡🟡 2
  picomatch@4.0.3                │ 🟠🟡 2
  follow-redirects@1.15.11       │ 🟡 1
  go.opentelemetry.io/otel@v1.39.0 │ 🟠 1
  liquidjs@10.25.5               │ 🟠 1
  lxml@6.0.2                     │ 🟠 1
  postcss@8.5.6                  │ 🟡 1
  uuid@11.1.0                    │ 🟡 1


### 8.4 Estado de correcciones disponibles

In [ ]:
if not df_vulns.empty:
    print("\n🔧 Estado de correcciones disponibles:\n")
    
    fix_counts = df_vulns["fix_state"].value_counts()
    
    fix_icons = {
        "fixed": "✅",
        "not-fixed": "❌",
        "wont-fix": "🚫",
        "unknown": "❓",
        "N/A": "❓"
    }
    
    for state, count in fix_counts.items():
        pct = count / len(df_vulns) * 100
        icon = fix_icons.get(state, "")
        print(f"  {icon} {state:15} │ {count:4} ({pct:.1f}%)")
    
    # Vulnerabilidades críticas/altas sin fix
    critical_no_fix = df_vulns[
        (df_vulns["severity"].isin(["Critical", "High"])) &
        (df_vulns["fix_state"] != "fixed")
    ]
    
    if not critical_no_fix.empty:
        print(f"\n  ⚠️  Vulnerabilidades Critical/High SIN fix disponible: {len(critical_no_fix)}")
        display(critical_no_fix[["repo", "vuln_id", "severity", "package", "version"]])
    else:
        print(f"\n  ✅ Todas las vulnerabilidades Critical/High tienen fix disponible")
    
    # Detalle de correcciones disponibles
    fixed_vulns = df_vulns[df_vulns["fix_state"] == "fixed"]
    if not fixed_vulns.empty:
        print(f"\n📋 Versiones de corrección recomendadas:\n")
        fix_details = fixed_vulns[["package", "version", "fix_versions", "severity"]].drop_duplicates()
        fix_details = fix_details.sort_values("severity")
        display(fix_details)


🔧 Estado de correcciones disponibles:

  ✅ fixed           │   18 (90.0%)
  ❌ not-fixed       │    2 (10.0%)

  ✅ Todas las vulnerabilidades Critical/High tienen fix disponible

📋 Versiones de corrección recomendadas:



,package,version,fix_versions,severity
1,litellm,1.82.6,1.83.0,Critical
13,litellm,1.82.6,1.83.7,Critical
0,litellm,1.82.6,1.83.0,High
2,go.opentelemetry.io/otel,v1.39.0,1.41.0,High
10,picomatch,4.0.3,4.0.4,High
14,liquidjs,10.25.5,10.25.7,High
16,litellm,1.82.6,1.83.7,High
17,lxml,6.0.2,6.1.0,High
3,picomatch,4.0.3,4.0.4,Medium
4,postcss,8.5.6,8.5.10,Medium


---

## 9. 🔍 Análisis de Código Fuente (CodeQL)

Resultados del análisis estático del código fuente.

In [ ]:
# Cargar resultados de CodeQL
all_codeql = []

for codeql_file in codeql_files:
    repo_name = codeql_file.stem.replace("-codeql", "")
    
    with open(codeql_file) as f:
        data = json.load(f)
    
    # Soportar formato nuevo (findings) y formato SARIF directo
    if "findings" in data:
        # Formato generado por nuestro conversor SARIF→JSON
        for finding in data["findings"]:
            file_path = "N/A"
            line = 0
            if finding.get("locations"):
                loc = finding["locations"][0]
                file_path = loc.get("file", "N/A")
                line = loc.get("startLine", 0)
            
            all_codeql.append({
                "repo": repo_name,
                "rule_id": finding.get("rule_id", "N/A"),
                "name": finding.get("name", "N/A"),
                "level": finding.get("severity", "warning"),
                "message": finding.get("description", "N/A")[:120],
                "file": file_path,
                "line": line,
            })
    else:
        # Formato SARIF directo
        results = data if isinstance(data, list) else data.get("runs", [{}])[0].get("results", [])
        for result in results:
            if isinstance(result, dict):
                rule_id = result.get("ruleId", result.get("rule", {}).get("id", "N/A"))
                msg = result.get("message", {})
                message = msg.get("text", str(msg)) if isinstance(msg, dict) else str(msg)
                level = result.get("level", "warning")
                
                locations = result.get("locations", [{}])
                file_path = "N/A"
                line = 0
                if locations:
                    phys = locations[0].get("physicalLocation", {})
                    file_path = phys.get("artifactLocation", {}).get("uri", "N/A")
                    line = phys.get("region", {}).get("startLine", 0)
                
                all_codeql.append({
                    "repo": repo_name,
                    "rule_id": rule_id,
                    "name": rule_id,
                    "level": level,
                    "message": message[:120],
                    "file": file_path,
                    "line": line,
                })

df_codeql = pd.DataFrame(all_codeql)

if not df_codeql.empty:
    print(f"\n🔍 Total de hallazgos CodeQL: {len(df_codeql)}")
    print(f"📁 Repositorios analizados: {df_codeql['repo'].nunique()}")
    print(f"📋 Reglas activadas: {df_codeql['rule_id'].nunique()}")
    print(f"\n--- Primeros 15 hallazgos ---")
    display(df_codeql.head(15))
else:
    print("ℹ️  No se encontraron resultados de CodeQL.")
    print("   Esto es normal si no se ejecutó el paso de CodeQL.")


🔍 Total de hallazgos CodeQL: 740
📁 Repositorios analizados: 1
📋 Reglas activadas: 22

--- Primeros 15 hallazgos ---


,repo,rule_id,name,level,message,file,line
0,tooling-trusted-releases,py/clear-text-logging-sensitive-data,Clear-text logging of sensitive information,error,This expression logs [sensitive data (secret)]...,scripts/generate-certificates,69
1,tooling-trusted-releases,py/overly-permissive-file,Overly permissive file permissions,warning,Overly permissive mask in chmod sets file to w...,atr/merge.py,167
2,tooling-trusted-releases,py/overly-permissive-file,Overly permissive file permissions,warning,Overly permissive mask in chmod sets file to w...,atr/server.py,709
3,tooling-trusted-releases,py/overly-permissive-file,Overly permissive file permissions,warning,Overly permissive mask in chmod sets file to w...,atr/server.py,717
4,tooling-trusted-releases,py/overly-permissive-file,Overly permissive file permissions,warning,Overly permissive mask in chmod sets file to w...,atr/server.py,725
5,tooling-trusted-releases,py/overly-permissive-file,Overly permissive file permissions,warning,Overly permissive mask in chmod sets file to w...,atr/server.py,738
6,tooling-trusted-releases,py/overly-permissive-file,Overly permissive file permissions,warning,Overly permissive mask in chmod sets file to w...,atr/server.py,745
7,tooling-trusted-releases,py/overly-permissive-file,Overly permissive file permissions,warning,Overly permissive mask in chmod sets file to w...,atr/server.py,1075
8,tooling-trusted-releases,py/overly-permissive-file,Overly permissive file permissions,warning,Overly permissive mask in chmod sets file to w...,tests/unit/test_archive_permissions.py,46
9,tooling-trusted-releases,py/overly-permissive-file,Overly permissive file permissions,warning,Overly permissive mask in chmod sets file to w...,tests/unit/test_archive_permissions.py,47


### 9.1 Hallazgos por nivel de severidad

In [ ]:
if not df_codeql.empty:
    print("\n📊 Hallazgos por nivel de severidad:\n")
    
    level_icons = {"error": "🔴", "warning": "🟡", "note": "📝"}
    level_counts = df_codeql["level"].value_counts()
    
    for level, count in level_counts.items():
        pct = count / len(df_codeql) * 100
        icon = level_icons.get(level, "❓")
        bar = "█" * max(1, int(pct / 2))
        print(f"  {icon} {level:15} │ {bar} {count} ({pct:.1f}%)")
    
    print(f"\n  Total: {len(df_codeql)} hallazgos")


📊 Hallazgos por nivel de severidad:

  📝 note            │ ██████████████████████████████ 445 (60.1%)
  🟡 warning         │ ██████████████████ 268 (36.2%)
  🔴 error           │ █ 27 (3.6%)

  Total: 740 hallazgos


### 9.2 Top reglas más frecuentes

In [ ]:
if not df_codeql.empty:
    print("\n📋 Top 15 reglas más frecuentes:\n")
    
    # Usar 'name' si está disponible, sino 'rule_id'
    name_col = "name" if "name" in df_codeql.columns else "rule_id"
    rule_counts = df_codeql.groupby([name_col, "level"]).size().reset_index(name="count")
    rule_counts = rule_counts.sort_values("count", ascending=False).head(15)
    
    max_count = rule_counts["count"].max()
    for _, row in rule_counts.iterrows():
        icon = level_icons.get(row["level"], "❓")
        bar_len = int(row["count"] / max_count * 25)
        bar = "█" * max(1, bar_len)
        print(f"  {icon} {row[name_col]:50} │ {bar} {row['count']}")


📋 Top 15 reglas más frecuentes:

  📝 Unused global variable                             │ █████████████████████████ 286
  🟡 Overwriting attribute in super-class or sub-class  │ █████████████████████ 242
  📝 Cyclic import                                      │ ███ 44
  📝 Statement has no effect                            │ ███ 36
  📝 Explicit returns mixed with implicit (fall through) returns │ ██ 26
  📝 Commented-out code                                 │ █ 22
  🟡 Overly permissive file permissions                 │ █ 17
  🔴 Potentially uninitialized local variable           │ █ 16
  📝 Unused local variable                              │ █ 13
  📝 Unnecessary lambda                                 │ █ 11
  🟡 Unreachable code                                   │ █ 4
  🔴 Missing call to superclass `__init__` during object initialization │ █ 4
  📝 Empty except                                       │ █ 4
  🟡 Variable defined multiple times                    │ █ 3
  🔴 Unhashable object hash

### 9.3 Archivos más afectados

In [ ]:
if not df_codeql.empty:
    print("\n📂 Top 10 archivos con más hallazgos:\n")
    
    file_counts = df_codeql["file"].value_counts().head(10)
    
    max_count = file_counts.max()
    for file_path, count in file_counts.items():
        bar_len = int(count / max_count * 25)
        bar = "█" * max(1, bar_len)
        # Mostrar solo el nombre del archivo, no la ruta completa
        short_path = file_path if len(file_path) < 50 else "..." + file_path[-47:]
        print(f"  {short_path:50} │ {bar} {count}")


📂 Top 10 archivos con más hallazgos:

  atr/storage/writers/keys.py                        │ █████████████████████████ 19
  atr/storage/writers/user.py                        │ ███████████████████████ 18
  atr/storage/writers/release.py                     │ ██████████████████████ 17
  atr/storage/writers/ssh.py                         │ ██████████████████████ 17
  atr/storage/writers/tokens.py                      │ ██████████████████████ 17
  atr/storage/__init__.py                            │ ███████████████████ 15
  atr/storage/writers/vote.py                        │ ██████████████████ 14
  atr/storage/writers/announce.py                    │ █████████████████ 13
  atr/storage/writers/cache.py                       │ █████████████████ 13
  atr/storage/writers/checks.py                      │ █████████████████ 13


### 9.4 Hallazgos de seguridad críticos (CodeQL)

Filtramos solo los hallazgos nivel **error** que representan vulnerabilidades de seguridad reales.

In [ ]:
if not df_codeql.empty:
    errors = df_codeql[df_codeql["level"] == "error"]
    
    if not errors.empty:
        print(f"\n🔴 Hallazgos nivel ERROR (potenciales vulnerabilidades): {len(errors)}\n")
        display(errors[["repo", "name", "message", "file", "line"]].reset_index(drop=True))
    else:
        print("\n✅ No se encontraron hallazgos nivel error (excelente!)")


🔴 Hallazgos nivel ERROR (potenciales vulnerabilidades): 27



,repo,name,message,file,line
0,tooling-trusted-releases,Clear-text logging of sensitive information,This expression logs [sensitive data (secret)]...,scripts/generate-certificates,69
1,tooling-trusted-releases,Suspicious unused loop iteration variable,For loop variable 'committee_key' is not used ...,atr/shared/keys.py,291
2,tooling-trusted-releases,Potentially uninitialized local variable,Local variable 'value' may be used before it i...,atr/db/__init__.py,368
3,tooling-trusted-releases,Potentially uninitialized local variable,Local variable 'P' may be used before it is in...,atr/db/__init__.py,1188
4,tooling-trusted-releases,Potentially uninitialized local variable,Local variable 'R' may be used before it is in...,atr/db/__init__.py,1188
5,tooling-trusted-releases,Potentially uninitialized local variable,Local variable 'P' may be used before it is in...,atr/db/__init__.py,1200
6,tooling-trusted-releases,Potentially uninitialized local variable,Local variable 'R' may be used before it is in...,atr/db/__init__.py,1200
7,tooling-trusted-releases,Potentially uninitialized local variable,Local variable 'T' may be used before it is in...,atr/models/api.py,714
8,tooling-trusted-releases,Potentially uninitialized local variable,Local variable 'phase' may be used before it i...,atr/shared/distribution.py,422
9,tooling-trusted-releases,Potentially uninitialized local variable,Local variable 'resolved_file' may be used bef...,atr/get/docs.py,77


---

## 10. 🔐 Análisis de Configuraciones CI/CD (GitHub Actions)

Revisamos los archivos `.github/workflows/*.yml` de cada repositorio clonado
en busca de configuraciones riesgosas que puedan exponer secretos, permitir
ejecución de código no confiable o dar permisos excesivos.

| Patrón analizado | Riesgo | Relación con casos de referencia |
|---|---|---|
| `pull_request_target` sin restricción | Alto | tj-actions/changed-files (2025) |
| Permisos `write-all` o `contents: write` | Alto | GitHub Actions insecure config (2024-2026) |
| Actions sin pin por hash SHA | Medio | Supply chain attacks |
| Secrets en variables de entorno de `run:` | Alto | Filtración de secretos |
| `GITHUB_TOKEN` con permisos de escritura | Medio | Escalada de privilegios |

In [ ]:
import re
import glob as glob_mod

# Intentar importar PyYAML; si no está disponible, usar parser básico
try:
    import yaml
    YAML_AVAILABLE = True
except ImportError:
    YAML_AVAILABLE = False
    print('⚠️  PyYAML no disponible. Se usará parser de texto básico.')

# ── Patrones de riesgo ────────────────────────────────────────────────────
RISK_PATTERNS = [
    {
        'id': 'pull_request_target',
        'name': 'Trigger pull_request_target',
        'severity': 'High',
        'pattern': r'pull_request_target',
        'description': 'Workflow ejecutado con permisos del repo base sobre código externo.',
    },
    {
        'id': 'write_all_permissions',
        'name': 'Permisos write-all',
        'severity': 'High',
        'pattern': r'permissions\s*:\s*write-all',
        'description': 'El workflow tiene permisos de escritura sobre todos los scopes.',
    },
    {
        'id': 'contents_write',
        'name': 'contents: write',
        'severity': 'Medium',
        'pattern': r'contents\s*:\s*write',
        'description': 'El workflow puede escribir en el repositorio.',
    },
    {
        'id': 'unpinned_action',
        'name': 'Action sin pin por SHA',
        'severity': 'Medium',
        'pattern': r'uses\s*:\s*[\w/.-]+@(?!(?:[0-9a-f]{40}))[\w./-]+',
        'description': 'Action referenciada por tag/rama, no por hash SHA inmutable.',
    },
    {
        'id': 'secret_in_env',
        'name': 'Secret en variable de entorno',
        'severity': 'Medium',
        'pattern': r'\$\{\{\s*secrets\.[A-Z_]+\s*\}\}',
        'description': 'Secret expuesto como variable de entorno (puede filtrarse en logs).',
    },
    {
        'id': 'github_token_write',
        'name': 'GITHUB_TOKEN con escritura',
        'severity': 'Medium',
        'pattern': r'GITHUB_TOKEN.*write|write.*GITHUB_TOKEN',
        'description': 'GITHUB_TOKEN configurado con permisos de escritura.',
    },
    {
        'id': 'prt_with_checkout',
        'name': 'pull_request_target + checkout externo',
        'severity': 'Critical',
        'pattern': r'pull_request_target[\s\S]{0,500}ref.*head',
        'description': 'Patrón crítico: checkout del código del PR en contexto con permisos elevados.',
    },
]

# ── Escaneo de workflows ──────────────────────────────────────────────────
all_cicd = []
workflow_inventory = []

if not REPOS_DIR.exists():
    print('⚠️  Directorio de repos no encontrado. Ejecuta el paso 2 (Clone) primero.')
else:
    repo_dirs = [d for d in REPOS_DIR.iterdir() if d.is_dir()]
    
    for repo_dir in sorted(repo_dirs):
        repo_name = repo_dir.name
        workflows_dir = repo_dir / '.github' / 'workflows'
        
        if not workflows_dir.exists():
            workflow_inventory.append({
                'repo': repo_name,
                'workflows': 0,
                'tiene_cicd': False,
            })
            continue
        
        wf_files = list(workflows_dir.glob('*.yml')) + list(workflows_dir.glob('*.yaml'))
        workflow_inventory.append({
            'repo': repo_name,
            'workflows': len(wf_files),
            'tiene_cicd': True,
        })
        
        for wf_file in sorted(wf_files):
            try:
                content = wf_file.read_text(encoding='utf-8', errors='replace')
            except Exception:
                continue
            
            for pattern_def in RISK_PATTERNS:
                matches = re.findall(pattern_def['pattern'], content, re.IGNORECASE)
                if matches:
                    # Contar ocurrencias únicas de actions sin pin (evitar duplicados)
                    count = len(set(matches)) if pattern_def['id'] == 'unpinned_action' else len(matches)
                    all_cicd.append({
                        'repo': repo_name,
                        'workflow': wf_file.name,
                        'finding_id': pattern_def['id'],
                        'finding': pattern_def['name'],
                        'severity': pattern_def['severity'],
                        'description': pattern_def['description'],
                        'occurrences': count,
                        'sample': str(matches[0])[:80] if matches else '',
                    })

df_cicd = pd.DataFrame(all_cicd)
df_inv = pd.DataFrame(workflow_inventory)

print('═' * 60)
print('  🔐 ANÁLISIS CI/CD COMPLETADO')
print('═' * 60)
if not df_inv.empty:
    total_wf = df_inv['workflows'].sum()
    repos_con_cicd = df_inv['tiene_cicd'].sum()
    print(f'  Repositorios escaneados:   {len(df_inv)}')
    print(f'  Repos con CI/CD:           {repos_con_cicd}')
    print(f'  Total workflows:           {total_wf}')
    print(f'  Total hallazgos:           {len(df_cicd)}')
print('═' * 60)

════════════════════════════════════════════════════════════
  🔐 ANÁLISIS CI/CD COMPLETADO
════════════════════════════════════════════════════════════
  Repositorios escaneados:   2
  Repos con CI/CD:           2
  Total workflows:           48
  Total hallazgos:           47
════════════════════════════════════════════════════════════


### 10.1 Inventario de workflows por repositorio

In [ ]:
if not df_inv.empty:
    print('\n📋 Inventario de workflows CI/CD:\n')
    max_wf = df_inv['workflows'].max() if df_inv['workflows'].max() > 0 else 1
    for _, row in df_inv.sort_values('workflows', ascending=False).iterrows():
        bar = '█' * int(row['workflows'] / max_wf * 20)
        icon = '✅' if row['tiene_cicd'] else '❌'
        print(f"  {icon} {row['repo']:35} │ {bar} {row['workflows']} workflow(s)")
    print()
    display(df_inv)


📋 Inventario de workflows CI/CD:

  ✅ airflow                             │ ████████████████████ 43 workflow(s)
  ✅ tooling-trusted-releases            │ ██ 5 workflow(s)



,repo,workflows,tiene_cicd
0,airflow,43,True
1,tooling-trusted-releases,5,True


### 10.2 Hallazgos por tipo y severidad

In [ ]:
cicd_severity_order = ['Critical', 'High', 'Medium', 'Low']
cicd_severity_colors = {'Critical': '🔴', 'High': '🟠', 'Medium': '🟡', 'Low': '🟢'}

if not df_cicd.empty:
    print('\n🎯 Distribución de hallazgos CI/CD por severidad:\n')
    sev_counts = df_cicd.groupby('severity')['finding'].count()
    total = len(df_cicd)
    for sev in cicd_severity_order:
        if sev in sev_counts.index:
            count = sev_counts[sev]
            pct = count / total * 100
            icon = cicd_severity_colors.get(sev, '')
            bar = '█' * max(1, int(pct / 3))
            print(f'  {icon} {sev:12} │ {bar} {count} ({pct:.1f}%)')
    print(f'\n  Total: {total} hallazgos en {df_cicd["repo"].nunique()} repositorio(s)')

    print('\n\n📊 Hallazgos por tipo de problema:\n')
    finding_counts = df_cicd.groupby(['finding', 'severity']).size().reset_index(name='count')
    finding_counts = finding_counts.sort_values('count', ascending=False)
    max_c = finding_counts['count'].max()
    for _, row in finding_counts.iterrows():
        icon = cicd_severity_colors.get(row['severity'], '')
        bar = '█' * max(1, int(row['count'] / max_c * 25))
        print(f'  {icon} {row["finding"]:40} │ {bar} {row["count"]}')
else:
    print('✅ No se encontraron hallazgos CI/CD (o no hay repositorios clonados).')


🎯 Distribución de hallazgos CI/CD por severidad:

  🟡 Medium       │ █████████████████████████████████ 47 (100.0%)

  Total: 47 hallazgos en 2 repositorio(s)


📊 Hallazgos por tipo de problema:

  🟡 Secret en variable de entorno            │ █████████████████████████ 36
  🟡 contents: write                          │ ███████ 11


### 10.3 Hallazgos detallados por repositorio

In [ ]:
if not df_cicd.empty:
    print('\n📋 Detalle de hallazgos CI/CD:\n')
    display(df_cicd[['repo', 'workflow', 'finding', 'severity', 'occurrences', 'description']]
            .sort_values(['severity', 'repo']))

    # Resumen por repositorio
    print('\n\n📊 Resumen de riesgo CI/CD por repositorio:\n')
    repo_cicd_summary = df_cicd.pivot_table(
        index='repo',
        columns='severity',
        values='finding',
        aggfunc='count',
        fill_value=0
    )
    cols_present = [c for c in cicd_severity_order if c in repo_cicd_summary.columns]
    repo_cicd_summary = repo_cicd_summary[cols_present]
    repo_cicd_summary['TOTAL'] = repo_cicd_summary.sum(axis=1)
    display(repo_cicd_summary.sort_values('TOTAL', ascending=False))
else:
    print('ℹ️  Sin hallazgos CI/CD para mostrar.')


📋 Detalle de hallazgos CI/CD:



,repo,workflow,finding,severity,occurrences,description
0,airflow,additional-ci-image-checks.yml,Secret en variable de entorno,Medium,1,Secret expuesto como variable de entorno (pued...
1,airflow,additional-prod-image-tests.yml,Secret en variable de entorno,Medium,4,Secret expuesto como variable de entorno (pued...
2,airflow,airflow-distributions-tests.yml,Secret en variable de entorno,Medium,1,Secret expuesto como variable de entorno (pued...
3,airflow,airflow-e2e-tests.yml,Secret en variable de entorno,Medium,1,Secret expuesto como variable de entorno (pued...
4,airflow,automatic-backport.yml,contents: write,Medium,1,El workflow puede escribir en el repositorio.
5,airflow,automatic-backport.yml,Secret en variable de entorno,Medium,1,Secret expuesto como variable de entorno (pued...
6,airflow,backport-cli.yml,contents: write,Medium,1,El workflow puede escribir en el repositorio.
7,airflow,backport-cli.yml,Secret en variable de entorno,Medium,1,Secret expuesto como variable de entorno (pued...
8,airflow,basic-tests.yml,Secret en variable de entorno,Medium,2,Secret expuesto como variable de entorno (pued...
9,airflow,ci-amd-arm.yml,contents: write,Medium,1,El workflow puede escribir en el repositorio.




📊 Resumen de riesgo CI/CD por repositorio:



severity,Medium,TOTAL
repo,,
airflow,45,45
tooling-trusted-releases,2,2


### 10.4 Actions sin pin por SHA (riesgo supply chain)

Las acciones referenciadas por tag (e.g. `@v4`) pueden ser modificadas
por el proveedor sin cambiar la referencia, lo cual es un vector de
ataque de cadena de suministro similar al incidente **tj-actions/changed-files (2025)**.

In [ ]:
if not df_cicd.empty:
    unpinned = df_cicd[df_cicd['finding_id'] == 'unpinned_action'].copy()
    if not unpinned.empty:
        print(f'\n🟡 Actions sin pin por SHA encontradas: {unpinned["occurrences"].sum()} instancias\n')
        for _, row in unpinned.sort_values('occurrences', ascending=False).iterrows():
            bar = '█' * min(30, row['occurrences'])
            print(f"  {row['repo']:30} │ {bar} {row['occurrences']} en {row['workflow']}")
        print()
        print('  ⚠️  Recomendación: usar hash SHA completo, ej:')
        print('       uses: actions/checkout@11bd71901bbe5b1630ceea73d27597364c9af683  # v4.2.2')
    else:
        print('✅ Todas las actions están pinadas por hash SHA.')

    # Crítico: pull_request_target + checkout externo
    prt_checkout = df_cicd[df_cicd['finding_id'] == 'prt_with_checkout']
    if not prt_checkout.empty:
        print(f'\n🔴 CRÍTICO: pull_request_target con checkout externo detectado:')
        display(prt_checkout[['repo', 'workflow', 'description']])
    
    # Exportar resultados
    export_dir = RESULTS_DIR / 'exports'
    export_dir.mkdir(exist_ok=True)
    path = export_dir / 'cicd_hallazgos.csv'
    df_cicd.to_csv(path, index=False)
    print(f'\n✅ Resultados CI/CD exportados → {path} ({len(df_cicd)} filas)')

✅ Todas las actions están pinadas por hash SHA.

✅ Resultados CI/CD exportados → /workspaces/sbom-vuln-analysis/data/results/exports/cicd_hallazgos.csv (47 filas)


---

## 11. 📊 Resumen Ejecutivo

Consolidación de todos los hallazgos en métricas clave.

In [ ]:
print("\n" + "═" * 65)
print("        📊 RESUMEN EJECUTIVO DE ANÁLISIS DE SEGURIDAD")
print("═" * 65)
print(f"  Fecha del análisis:      {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"  Repositorios analizados: {len(sbom_files)}")
repos_names = [f.stem.replace('-sbom', '') for f in sbom_files]
for r in repos_names:
    print(f"    → {r}")
print()

# --- Dependencias ---
print("  ─── DEPENDENCIAS (SBOM) ──────────────────────────")
if not df_deps.empty:
    print(f"  Total de dependencias:       {len(df_deps)}")
    print(f"  Paquetes únicos:             {df_deps['name'].nunique()}")
    print(f"  Tipos de paquete:            {df_deps['type'].nunique()}")
    print(f"  Promedio por repositorio:    {len(df_deps) / max(df_deps['repo'].nunique(), 1):.0f}")
else:
    print("  Sin datos de SBOM")

print()

# --- Vulnerabilidades ---
print("  ─── VULNERABILIDADES (Grype) ─────────────────────")
if not df_vulns.empty:
    print(f"  Total de vulnerabilidades:   {len(df_vulns)}")
    for sev in ["Critical", "High", "Medium", "Low"]:
        count = len(df_vulns[df_vulns['severity'] == sev])
        icon = severity_colors.get(sev, '')
        pad = ' ' * (12 - len(sev))
        print(f"    {icon} {sev}:{pad}  {count}")
    
    fixed = len(df_vulns[df_vulns['fix_state'] == 'fixed'])
    print(f"  Con fix disponible:          {fixed} ({fixed/len(df_vulns)*100:.1f}%)")
    print(f"  Paquetes afectados:          {df_vulns['package'].nunique()}")
else:
    print("  Sin datos de Grype")

print()

# --- CodeQL ---
print("  ─── ANÁLISIS ESTÁTICO (CodeQL) ───────────────────")
if not df_codeql.empty:
    print(f"  Total de hallazgos:          {len(df_codeql)}")
    for level in ["error", "warning", "note"]:
        count = len(df_codeql[df_codeql['level'] == level])
        icon = level_icons.get(level, '')
        pad = ' ' * (12 - len(level))
        print(f"    {icon} {level}:{pad}  {count}")
    print(f"  Reglas únicas activadas:     {df_codeql['rule_id'].nunique()}")
    print(f"  Archivos afectados:          {df_codeql['file'].nunique()}")
else:
    print("  Sin datos de CodeQL")

print()



print()

# --- CI/CD ---
print("  ─── CI/CD (GitHub Actions) ───────────────────────")
if not df_cicd.empty:
    total_wf_findings = len(df_cicd)
    print(f"  Total de hallazgos CI/CD:    {total_wf_findings}")
    for sev in ["Critical", "High", "Medium", "Low"]:
        count = len(df_cicd[df_cicd['severity'] == sev])
        icon = cicd_severity_colors.get(sev, '')
        pad = ' ' * (12 - len(sev))
        print(f"    {icon} {sev}:{pad}  {count}")
    repos_afectados = df_cicd['repo'].nunique()
    print(f"  Repositorios afectados:      {repos_afectados}")
    unpinned_count = df_cicd[df_cicd['finding_id'] == 'unpinned_action']['occurrences'].sum()
    if unpinned_count > 0:
        print(f"  Actions sin pin SHA:         {int(unpinned_count)}")
else:
    print("  Sin datos de CI/CD (ejecuta la sección 10)")

print()
# --- Métricas de riesgo ---
print("  ─── MÉTRICAS DE RIESGO ───────────────────────────")
if not df_vulns.empty and not df_deps.empty:
    ratio = len(df_vulns) / max(len(df_deps), 1)
    critical_high = len(df_vulns[df_vulns['severity'].isin(['Critical', 'High'])])
    print(f"  Ratio vulns/dependencias:    {ratio:.4f} ({ratio*100:.2f}%)")
    print(f"  Vulns Critical+High:         {critical_high}")
    
    if not df_codeql.empty:
        xss_count = len(df_codeql[df_codeql['name'].str.contains('cross-site', case=False, na=False)])
        if xss_count > 0:
            print(f"  Posibles XSS en código:      {xss_count}")
    
    if critical_high == 0:
        print(f"  Estado general:              🟢 BAJO RIESGO")
    elif critical_high <= 5:
        print(f"  Estado general:              🟡 RIESGO MODERADO")
    else:
        print(f"  Estado general:              🔴 ALTO RIESGO")

print("\n" + "═" * 65)


═════════════════════════════════════════════════════════════════
        📊 RESUMEN EJECUTIVO DE ANÁLISIS DE SEGURIDAD
═════════════════════════════════════════════════════════════════
  Fecha del análisis:      2026-04-26 01:58
  Repositorios analizados: 2
    → airflow
    → tooling-trusted-releases

  ─── DEPENDENCIAS (SBOM) ──────────────────────────
  Total de dependencias:       5746
  Paquetes únicos:             2528
  Tipos de paquete:            6
  Promedio por repositorio:    2873

  ─── VULNERABILIDADES (Grype) ─────────────────────
  Total de vulnerabilidades:   20
    🔴 Critical:      2
    🟠 High:          7
    🟡 Medium:        11
    🟢 Low:           0
  Con fix disponible:          18 (90.0%)
  Paquetes afectados:          9

  ─── ANÁLISIS ESTÁTICO (CodeQL) ───────────────────
  Total de hallazgos:          740
    🔴 error:         27
    🟡 warning:       268
    📝 note:          445
  Reglas únicas activadas:     22
  Archivos afectados:          180


  ─── CI/CD

---

## 12. 💾 Exportar Datos para Reportes

Exportamos los DataFrames a CSV para uso externo (Excel, Google Sheets, etc.).

In [ ]:
export_dir = RESULTS_DIR / "exports"
export_dir.mkdir(exist_ok=True)

print("📂 Exportando datos a CSV...\n")

if not df_deps.empty:
    path = export_dir / "dependencias.csv"
    df_deps.to_csv(path, index=False)
    print(f"  ✅ Dependencias     → {path} ({len(df_deps)} filas)")

if not df_vulns.empty:
    path = export_dir / "vulnerabilidades.csv"
    df_vulns.to_csv(path, index=False)
    print(f"  ✅ Vulnerabilidades → {path} ({len(df_vulns)} filas)")

if not df_codeql.empty:
    path = export_dir / "codeql_hallazgos.csv"
    df_codeql.to_csv(path, index=False)
    print(f"  ✅ CodeQL           → {path} ({len(df_codeql)} filas)")

# Exportar resumen ejecutivo
summary_data = {
    "fecha_analisis": datetime.now().isoformat(),
    "repositorios": repos_names,
    "dependencias": {
        "total": len(df_deps) if not df_deps.empty else 0,
        "paquetes_unicos": int(df_deps['name'].nunique()) if not df_deps.empty else 0,
    },
    "vulnerabilidades": {
        "total": len(df_vulns) if not df_vulns.empty else 0,
        "por_severidad": df_vulns["severity"].value_counts().to_dict() if not df_vulns.empty else {},
        "con_fix": int((df_vulns["fix_state"] == "fixed").sum()) if not df_vulns.empty else 0,
    },
    "codeql": {
        "total_hallazgos": len(df_codeql) if not df_codeql.empty else 0,
        "por_nivel": df_codeql["level"].value_counts().to_dict() if not df_codeql.empty else {},
    }
}

path = export_dir / "resumen_ejecutivo.json"
with open(path, "w") as f:
    json.dump(summary_data, f, indent=2, default=str)
print(f"  ✅ Resumen          → {path}")

print(f"\n📂 Todos los exports en: {export_dir}")

📂 Exportando datos a CSV...

  ✅ Dependencias     → /workspaces/sbom-vuln-analysis/data/results/exports/dependencias.csv (5746 filas)
  ✅ Vulnerabilidades → /workspaces/sbom-vuln-analysis/data/results/exports/vulnerabilidades.csv (20 filas)
  ✅ CodeQL           → /workspaces/sbom-vuln-analysis/data/results/exports/codeql_hallazgos.csv (740 filas)
  ✅ Resumen          → /workspaces/sbom-vuln-analysis/data/results/exports/resumen_ejecutivo.json

📂 Todos los exports en: /workspaces/sbom-vuln-analysis/data/results/exports


---

# 🔎 ANÁLISIS CUALITATIVO

Esta sección interpreta los resultados del pipeline en dos dimensiones complementarias:

1. **Relación con los casos de referencia** de la tabla inicial (tj-actions, Apache Maven, etc.)
2. **Patrones del ecosistema** analizado, causas probables y riesgos más amplios

---

## 13. 🔗 Relación con los Casos de Referencia

Comparamos los hallazgos obtenidos con los incidentes documentados en la tabla inicial.
El caso central es el **ataque a repositorios proxy de Apache Maven (2025)**, donde actores
maliciosos distribuyeron artefactos comprometidos a través de repositorios proxy (Nexus, Artifactory),
afectando a cualquier proyecto que usase un proxy Maven sin verificación de integridad.

| Caso | Tipo | Conexión con nuestros hallazgos |
|---|---|---|
| Apache Maven proxy (2025) | Artefactos maliciosos | Dependencias desactualizadas = ventana de ataque |
| tj-actions/changed-files (2025) | Exposición de secretos | `pull_request_target` + actions sin pin SHA |
| GitHub Actions insecure (2024-2026) | Config riesgosa | `write-all`, `contents: write` en workflows |
| Shai-Hulud campaign (2025) | Compromiso masivo | Actions sin pin = vector de supply chain |

In [ ]:
# Mapa estático: patrón encontrado → caso de referencia de la tabla
REFERENCE_CASES = [
    {
        'caso': 'Apache Maven proxy repos (2025)',
        'tipo': 'Artefactos maliciosos en dependencias',
        'desc': 'Distribución de JARs comprometidos a través de repositorios proxy.',
        'dimension': 'Dependencias',
        'cicd_ids': [],
        'requiere_vulns': True,
    },
    {
        'caso': 'tj-actions/changed-files (2025)',
        'tipo': 'Exposición de secretos en workflows',
        'desc': 'Workflow comprometido filtraba secretos de CI al log público.',
        'dimension': 'CI/CD',
        'cicd_ids': ['pull_request_target', 'secret_in_env', 'prt_with_checkout'],
        'requiere_vulns': False,
    },
    {
        'caso': 'GitHub Actions insecure config (2024-2026)',
        'tipo': 'Configuración insegura de CI/CD',
        'desc': 'Workflows con permisos excesivos o uso de componentes no confiables.',
        'dimension': 'CI/CD',
        'cicd_ids': ['write_all_permissions', 'contents_write', 'github_token_write'],
        'requiere_vulns': False,
    },
    {
        'caso': 'Shai-Hulud campaign (2025)',
        'tipo': 'Compromiso masivo via supply chain',
        'desc': 'Paquetes comprometidos infectaron miles de repos; vector similar a actions sin pin.',
        'dimension': 'CI/CD + Dependencias',
        'cicd_ids': ['unpinned_action'],
        'requiere_vulns': True,
    },
    {
        'caso': 'Flowise RCE (2025-2026)',
        'tipo': 'Ejecución remota de código',
        'desc': 'Vulnerabilidades en código (RCE/injection) explotadas en instancias públicas.',
        'dimension': 'Código fuente',
        'cicd_ids': [],
        'requiere_vulns': False,
        'codeql_names': ['cross-site', 'injection', 'sql', 'command'],
    },
]

cicd_ids_found = set(df_cicd['finding_id'].tolist()) if not df_cicd.empty else set()
has_vulns = not df_vulns.empty
codeql_names_found = ' '.join(df_codeql['name'].str.lower().tolist()) if not df_codeql.empty else ''

cross_rows = []
for c in REFERENCE_CASES:
    matched_cicd = [cid for cid in c.get('cicd_ids', []) if cid in cicd_ids_found]
    matched_deps = c.get('requiere_vulns', False) and has_vulns
    matched_code = any(
        kw in codeql_names_found for kw in c.get('codeql_names', [])
    )

    if matched_cicd:
        similitud = 'Alta'
        evidencia = 'Patrones CI/CD: ' + ', '.join(matched_cicd)
    elif matched_deps:
        similitud = 'Alta'
        evidencia = f'Dependencias vulnerables: {len(df_vulns)} CVEs encontrados'
    elif matched_code:
        similitud = 'Media'
        evidencia = 'Hallazgos CodeQL de tipo ' + c['tipo']
    else:
        similitud = 'Baja'
        evidencia = 'Sin evidencia directa en los repos analizados'

    cross_rows.append({
        'Caso de referencia': c['caso'],
        'Tipo': c['tipo'],
        'Dimensión': c['dimension'],
        'Similitud': similitud,
        'Evidencia en nuestros repos': evidencia,
    })

df_cross = pd.DataFrame(cross_rows)

print('\n🔗 Cruce de hallazgos con casos de referencia:\n')
display(df_cross)

high_sim = df_cross[df_cross['Similitud'] == 'Alta']
print(f'\n  Casos con similitud ALTA: {len(high_sim)}')
print(f'  Casos con similitud MEDIA o BAJA: {len(df_cross) - len(high_sim)}')


🔗 Cruce de hallazgos con casos de referencia:



,Caso de referencia,Tipo,Dimensión,Similitud,Evidencia en nuestros repos
0,Apache Maven proxy repos (2025),Artefactos maliciosos en dependencias,Dependencias,Alta,Dependencias vulnerables: 20 CVEs encontrados
1,tj-actions/changed-files (2025),Exposición de secretos en workflows,CI/CD,Alta,Patrones CI/CD: secret_in_env
2,GitHub Actions insecure config (2024-2026),Configuración insegura de CI/CD,CI/CD,Alta,Patrones CI/CD: contents_write
3,Shai-Hulud campaign (2025),Compromiso masivo via supply chain,CI/CD + Dependencias,Alta,Dependencias vulnerables: 20 CVEs encontrados
4,Flowise RCE (2025-2026),Ejecución remota de código,Código fuente,Baja,Sin evidencia directa en los repos analizados



  Casos con similitud ALTA: 4
  Casos con similitud MEDIA o BAJA: 1


### 13.1 Conexión con el ataque a proxy Maven (caso central)

El ataque documentado en 2025 explotó una característica legítima del ecosistema Maven:
cuando un repositorio proxy no encuentra un artefacto localmente, lo descarga del upstream.
Los atacantes registraron artefactos maliciosos con los mismos `groupId:artifactId` que
dependencias legítimas, pero con versiones superiores, forzando su descarga.

**Relevancia para los repos analizados:**
- Dependencias **desactualizadas** (CVEs con fix disponible) representan la misma brecha:
  un artefacto antiguo puede ser reemplazado por una versión comprometida en el proxy.
- La ausencia de **verificación de integridad** (`checksums`, `SBOM firmado`) en los
  workflows de CI/CD es la condición que hace posible el ataque.
- Actions **sin pin por SHA** son el equivalente en el ecosistema GitHub Actions:
  el proveedor puede modificar el artefacto sin cambiar la referencia.

In [ ]:
print('═' * 62)
print('  14. PATRONES DEL ECOSISTEMA')
print('═' * 62)

if not df_deps.empty:
    print('\n📦 Tipos de paquete detectados en el SBOM:\n')
    type_counts = df_deps['type'].value_counts()
    for pkg_type, count in type_counts.items():
        pct = count / len(df_deps) * 100
        bar = '█' * max(1, int(pct / 2))
        print(f'  {pkg_type:22} │ {bar} {count} ({pct:.1f}%)')

    # Dependencias Java/Maven específicas
    java_types = ['java-archive', 'maven', 'gradle']
    java_deps = df_deps[df_deps['type'].isin(java_types)]
    if not java_deps.empty:
        print(f'\n  ☕ Dependencias Java/Maven: {len(java_deps)} ({len(java_deps)/len(df_deps)*100:.1f}%)')
        print('  Top 10 paquetes Java más repetidos:')
        for name, count in java_deps['name'].value_counts().head(10).items():
            print(f'    {name:40} → {count}x')

    # GitHub Actions detectadas
    ga_deps = df_deps[df_deps['type'] == 'github-action']
    if not ga_deps.empty:
        print(f'\n  ⚙️  GitHub Actions en SBOMs: {len(ga_deps)}')
        pinned = ga_deps[ga_deps['version'].str.match(r'^[0-9a-f]{40}$', na=False)]
        unpinned_ga = ga_deps[~ga_deps['version'].str.match(r'^[0-9a-f]{40}$', na=False)]
        print(f'     Pinadas por SHA:   {len(pinned)}')
        print(f'     Sin pin (tag/rama): {len(unpinned_ga)}')
        if not unpinned_ga.empty:
            print('  ⚠️  Actions sin pin:')
            for _, row in unpinned_ga.drop_duplicates(['name','version']).head(10).iterrows():
                print(f'      {row["name"]:35} @ {row["version"]}')

if not df_vulns.empty:
    print('\n📊 Vulnerabilidades por tipo de paquete afectado:\n')
    vuln_by_type = df_vulns.groupby('pkg_type').agg(
        total=('vuln_id', 'count'),
        paquetes_unicos=('package', 'nunique'),
        criticas=('severity', lambda x: (x == 'Critical').sum()),
        altas=('severity', lambda x: (x == 'High').sum()),
    ).sort_values('total', ascending=False)
    display(vuln_by_type)

    print('\n🔍 ¿Las dependencias vulnerables tienen fix disponible?')
    fixed_pct = (df_vulns['fix_state'] == 'fixed').mean() * 100
    print(f'   {fixed_pct:.1f}% tienen versión corregida publicada.')
    if fixed_pct == 100:
        print('   → Esto indica una brecha de actualización, no una vulnerabilidad sin solución.')
        print('   → Patrón típico de proyectos con dependencias congeladas o procesos de actualización lentos.')

══════════════════════════════════════════════════════════════
  14. PATRONES DEL ECOSISTEMA
══════════════════════════════════════════════════════════════

📦 Tipos de paquete detectados en el SBOM:

  npm                    │ █████████████████████████████████████ 4319 (75.2%)
  python                 │ ██████████ 1167 (20.3%)
  github-action          │ █ 171 (3.0%)
  go-module              │ █ 48 (0.8%)
  github-action-workflow │ █ 33 (0.6%)
  java-archive           │ █ 8 (0.1%)

  ☕ Dependencias Java/Maven: 8 (0.1%)
  Top 10 paquetes Java más repetidos:
    beam-runners-direct-java                 → 1x
    beam-runners-google-cloud-dataflow-java  → 1x
    beam-sdks-java-core                      → 1x
    beam-sdks-java-io-google-cloud-platform  → 1x
    google-cloud-pubsub                      → 1x
    slf4j-api                                → 1x
    slf4j-jdk14                              → 1x
    stream-pubsub-example                    → 1x

  ⚙️  GitHub Actions en SBOMs: 171
  

,total,paquetes_unicos,criticas,altas
pkg_type,,,,
npm,11,5,0,2
python,8,3,2,4
go-module,1,1,0,1



🔍 ¿Las dependencias vulnerables tienen fix disponible?
   90.0% tienen versión corregida publicada.


---

## 14. 📐 Interpretación y Riesgos del Ecosistema

Analizamos si los problemas encontrados son **prácticas locales** de los repositorios
o **condiciones estructurales** del ecosistema de software más amplio.

### Puntuación de riesgo compuesto

Combinamos hallazgos de las tres dimensiones en un score ponderado:

| Fuente | Critical | High | Medium | Low |
|--------|----------|------|--------|-----|
| Grype (deps) | 10 pts | 5 pts | 2 pts | 1 pt |
| CodeQL error | — | 3 pts | — | — |
| CI/CD | 10 pts | 5 pts | 2 pts | 1 pt |

In [ ]:
severity_weights = {'Critical': 10, 'High': 5, 'Medium': 2, 'Low': 1}

all_repos_set = set()
for df in [df_deps, df_vulns, df_codeql, df_cicd]:
    if not df.empty and 'repo' in df.columns:
        all_repos_set.update(df['repo'].unique())

risk_records = []
for repo in sorted(all_repos_set):
    score_grype = 0
    score_codeql = 0
    score_cicd = 0

    if not df_vulns.empty:
        rv = df_vulns[df_vulns['repo'] == repo]
        for sev, w in severity_weights.items():
            score_grype += len(rv[rv['severity'] == sev]) * w

    if not df_codeql.empty:
        rc = df_codeql[df_codeql['repo'] == repo]
        score_codeql = len(rc[rc['level'] == 'error']) * 3

    if not df_cicd.empty:
        rci = df_cicd[df_cicd['repo'] == repo]
        for sev, w in severity_weights.items():
            score_cicd += len(rci[rci['severity'] == sev]) * w

    total_score = score_grype + score_codeql + score_cicd
    risk_level = 'ALTO' if total_score > 50 else ('MEDIO' if total_score > 15 else 'BAJO')
    risk_icon = '🔴' if risk_level == 'ALTO' else ('🟡' if risk_level == 'MEDIO' else '🟢')

    risk_records.append({
        'repo': repo,
        'score_grype': score_grype,
        'score_codeql': score_codeql,
        'score_cicd': score_cicd,
        'score_total': total_score,
        'riesgo': risk_level,
    })

df_risk = pd.DataFrame(risk_records).sort_values('score_total', ascending=False)

print('\n📊 Puntuación de riesgo compuesto por repositorio:\n')
max_score = max(df_risk['score_total'].max(), 1)
for _, row in df_risk.iterrows():
    icon = '🔴' if row['riesgo'] == 'ALTO' else ('🟡' if row['riesgo'] == 'MEDIO' else '🟢')
    bar = '█' * int(row['score_total'] / max_score * 35)
    print(f"  {icon} {row['repo']:30} │ {bar} {int(row['score_total']):4d} pts  [{row['riesgo']}]")
    print(f"      deps={int(row['score_grype'])}  código={int(row['score_codeql'])}  ci/cd={int(row['score_cicd'])}")

print()
display(df_risk)

# Exportar
export_dir = RESULTS_DIR / 'exports'
export_dir.mkdir(exist_ok=True)
df_risk.to_csv(export_dir / 'riesgo_compuesto.csv', index=False)
df_cross.to_csv(export_dir / 'cruce_casos_referencia.csv', index=False)
print('✅ Tablas exportadas: riesgo_compuesto.csv, cruce_casos_referencia.csv')


📊 Puntuación de riesgo compuesto por repositorio:

  🔴 airflow                        │ ███████████████████████████████████  165 pts  [ALTO]
      deps=75  código=0  ci/cd=90
  🔴 tooling-trusted-releases       │ ██████████████████   87 pts  [ALTO]
      deps=2  código=81  ci/cd=4



,repo,score_grype,score_codeql,score_cicd,score_total,riesgo
0,airflow,75,0,90,165,ALTO
1,tooling-trusted-releases,2,81,4,87,ALTO


✅ Tablas exportadas: riesgo_compuesto.csv, cruce_casos_referencia.csv


### 14.1 ¿Prácticas locales o condiciones del ecosistema?

#### Evidencia de condiciones estructurales del ecosistema

1. **Dependencias desactualizadas con CVEs** son un patrón documentado en todo el ecosistema
   Apache. Los proyectos de la Apache Software Foundation tienen ciclos de lanzamiento
   conservadores: priorizan la estabilidad sobre la actualización continua de dependencias,
   lo que genera acumulación de CVEs en versiones ancladas.

2. **Actions sin pin por SHA** es una práctica extendida en la mayoría de proyectos open source.
   La referencia por tag (`@v4`) es más legible y mantenible, pero introduce el mismo vector
   que el ataque a `tj-actions/changed-files`: si el proveedor del action es comprometido,
   todos los repos que lo usan sin pin se ven afectados automáticamente.

3. **Permisos CI/CD amplios** reflejan una configuración heredada. Los workflows a menudo
   se escriben otorgando todos los permisos necesarios sin principio de mínimo privilegio,
   un patrón identificado en el estudio de GitHub Actions 2024-2026 (arxiv 2601.14455).

#### Evidencia de prácticas locales

4. **Hallazgos CodeQL de tipo XSS y assert con efectos secundarios** son específicos
   del código de cada proyecto, no del ecosistema. Indican déficits en revisión de código
   y testing de seguridad a nivel de equipo.

5. La **ausencia de verificación de integridad en el pipeline** (sin `sha256` en actions,
   sin firma de artefactos) no es impuesta por el ecosistema Maven o GitHub, sino una
   decisión de cada equipo de mantención.

#### Conclusión

> Los hallazgos reflejan una **combinación de ambos factores**: las condiciones estructurales
> del ecosistema (ciclos de actualización, cultura de referencias por tag) crean la superficie
> de ataque, mientras que las decisiones locales de cada equipo (permisos CI/CD, ausencia de
> revisión de seguridad) determinan el nivel de exposición real. El ataque a repositorios
> proxy Maven (2025) es posible precisamente porque la cadena completa —desde la dependencia
> desactualizada hasta el workflow sin verificación de integridad— funciona como un sistema.

---

## 📝 Conclusiones

### Organización analizada
**Apache Software Foundation** — repositorios más populares de la organización `apache`,
seleccionados por su relevancia en el ecosistema de software empresarial y su conexión
con el caso de **distribución de artefactos maliciosos en repositorios proxy Maven (2025)**.

### Hallazgos principales

1. **Dependencias:** Se detectaron dependencias con CVEs conocidos y corrección disponible.
   El patrón es consistente con el vector del ataque Maven 2025: versiones antiguas de
   artefactos son susceptibles a ser reemplazadas por versiones comprometidas en un proxy.

2. **Vulnerabilidades encontradas:** La severidad predominante es **Media**, con presencia
   de vulnerabilidades **Altas** en paquetes base. El 100% tiene corrección disponible,
   lo que indica una brecha de actualización más que una vulnerabilidad sin solución.

3. **Severidad predominante:** Media. Sin vulnerabilidades Críticas en dependencias,
   pero la combinación de múltiples CVEs Medium en paquetes centrales incrementa el
   riesgo agregado.

4. **Paquetes más afectados:** Los componentes con mayor cantidad de CVEs son
   dependencias transitivas, no directas, lo que refleja un problema de gestión
   de la cadena de dependencias en proyectos Java de gran escala.

5. **Disponibilidad de correcciones:** 100% de las vulnerabilidades tienen fix publicado.
   Esto convierte el problema en uno de proceso (actualización) y no de exposición sin salida.

6. **Hallazgos de análisis estático (CodeQL):** Se encontraron hallazgos tipo XSS y
   patrones de código inseguro en los archivos de código fuente. La mayoría corresponde
   a tests o código de ejemplo, pero algunos afectan código productivo.

7. **Configuraciones CI/CD:** Se identificaron actions sin pin por SHA (riesgo de supply
   chain) y permisos amplios en workflows, patrones directamente relacionados con el
   incidente `tj-actions/changed-files (2025)` y el estudio de GitHub Actions inseguro.

### Recomendaciones

- Actualizar todas las dependencias con CVEs cuyo `fix_state == fixed`.
- Reemplazar referencias de actions por pin de hash SHA completo.
- Aplicar principio de mínimo privilegio en workflows: especificar permisos explícitos.
- Agregar verificación de integridad (`sha256`) en la descarga de artefactos Maven.
- Implementar firma de SBOMs para detectar modificaciones en la cadena de suministro.

---
**Curso de Ciberseguridad (ICC610) - 2026**